In [1]:
from pydantic import BaseModel
from openai import OpenAI
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from src.rarekg.pubmed.pubmed_central import get_pmc_fulltext
from src.rarekg.sgr import (extract_entities, ExtractionResult)


# pmcid = "PMC9314610"

# title, abstract, para = get_pmc_fulltext(pmcid)
# print(title)
# print(abstract)







In [2]:
from typing import Set, Tuple, Union, Dict, Any
import json


def _normalize_entity_name(name: str) -> str:
    """
    Very simple text normalization for comparison:
      - lowercase
      - strip leading/trailing whitespace
      - collapse internal whitespace

    You can later extend this (e.g. Greek letters, hyphens, punctuation).
    """
    name = name.strip().lower()
    # collapse multiple spaces/tabs/newlines
    parts = name.split()
    return " ".join(parts)


def _entities_from_json_like(
    data: Union[str, Dict[str, Any], ExtractionResult]
) -> Set[Tuple[str, str]]:
   
    if isinstance(data, ExtractionResult):
        ents = data.entities
        result: Set[Tuple[str, str]] = set()
        for e in ents:
            if hasattr(e.type, "value"):
                etype = e.type.value
            else:
                etype = str(e.type)
                if "." in etype:
                    etype = etype.split(".")[-1]

            if etype == "gene":
                name = e.name              # no normalization
            else:
                name = _normalize_entity_name(e.name)
            result.add((name, etype))
        return result
    
    if isinstance(data, str):
        data = json.loads(data)

    entities = data.get("entities", [])
    result: Set[Tuple[str, str]] = set()
    for ent in entities:
        name = ent.get("name")
        etype = ent.get("type")
        if not name or not etype:
            continue
        if etype == "gene":
            norm_name = name  # keep as is
        else:
            norm_name = _normalize_entity_name(name)

        result.add((norm_name, str(etype)))
    return result


def evaluate_extraction(
    gold: Union[str, Dict[str, Any], ExtractionResult],
    pred: Union[str, Dict[str, Any], ExtractionResult],
) -> Dict[str, float]:
    gold_set = _entities_from_json_like(gold)
    pred_set = _entities_from_json_like(pred)

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    num_gold = len(gold_set)
    num_pred = len(pred_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "true_positives": float(tp),
        "false_positives": float(fp),
        "false_negatives": float(fn),
        "num_gold": float(num_gold),
        "num_pred": float(num_pred),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


In [3]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

In [4]:
import re

# Very simple pattern: parentheses with at least one 4-digit year
CITATION_RE = re.compile(
    r"\((?:[^()]*?\d{4}[^()]*)\)"
)

def strip_inline_citations(text: str) -> str:
    return CITATION_RE.sub("", text)

ETAL_RE = re.compile(r"\b[A-Z][a-z]+ et al\.\b")

def strip_citation_like_bits(text: str) -> str:
    text = CITATION_RE.sub("", text)
    text = ETAL_RE.sub("", text)
    return text


In [5]:
pmcids = ["PMC9437135", "PMC8901527" , "PMC12021325", "PMC4289844", "PMC10032320", "PMC10401045", "PMC9314610", "PMC9167473", "PMC7531107", "PMC2783042", "PMC9278302", "PMC7702659", "PMC3014844", "PMC5717851", "PMC8488375" ]

In [6]:
pmids = ["35655331", "34826654", "39435720", "25578972", "36803942", "37547106", "35238134", "35721635", "33004012", "19946579", "35452508", "32618441", "21209736", "29208045", "34616357"]

In [38]:
import requests
from typing import List, Dict, Any

PUBTATOR_EXPORT_URL = (
    "https://www.ncbi.nlm.nih.gov/research/pubtator3-api/publications/export/biocjson"
)


def fetch_pubtator_docs(pmids: List[str], full: bool = True) -> List[Dict[str, Any]]:
    """
    Fetch BioC-JSON annotations from PubTator3 for a list of PMIDs.
    """
    params = {
        "pmids": ",".join(str(p) for p in pmids),
    }
    if full:
        params["full"] = "true"  # include full text when available

    resp = requests.get(PUBTATOR_EXPORT_URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    # PubTator3 usually wraps docs under top-level key "PubTator3"
    if isinstance(data, dict) and "PubTator3" in data:
        return data["PubTator3"]
    elif isinstance(data, list):
        return data
    else:
        raise ValueError(f"Unexpected response shape: {type(data)}")


# def parse_entities_from_doc(doc: Dict[str, Any]) -> Dict[str, list]:
#     """
#     Extract genes, diseases, chemicals (drugs), and variants from a single document.
#     Returns lists of dicts with surface text, id, section, offsets, etc.
#     """
#     entities = {
#         "genes": [],
#         "diseases": [],
#         "chemicals": [],
#     }

#     for passage in doc.get("passages", []):
#         section = passage.get("infons", {}).get("section_type", "")
#         for ann in passage.get("annotations", []):
#             text = ann.get("text")
#             if not text:
#                 continue

#             infons = ann.get("infons", {})
#             ent_type = infons.get("type")  # e.g. Gene, Disease, Chemical, Variant
#             identifier = infons.get("identifier")  # e.g. NCBIGene:672, MESH:D009369, etc.

#             locs = ann.get("locations", [])
#             start = locs[0]["offset"] if locs else None
#             length = locs[0]["length"] if locs else None

#             record = {
#                 "text": text,
#                 "identifier": identifier,
#                 "raw_type": ent_type,
#                 "section": section,
#                 "offset": start,
#                 "length": length,
#             }

#             if ent_type in ("Gene", "Protein"):
#                 entities["genes"].append(record)
#             elif ent_type == "Disease":
#                 entities["diseases"].append(record)
#             elif ent_type in ("Chemical", "Drug"):
#                 entities["chemicals"].append(record)

#     return entities
from typing import Dict, List, Any


def _normalize_mention(text: str) -> str:
    """Normalize disease/chemical text for deduplication."""
    # lowercase + collapse multiple spaces
    return " ".join(text.lower().split())


def parse_entities_from_doc(doc: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    """
    Extract genes, diseases, and chemicals (drugs) from a single PubTator3 document.

    - Genes: keep as-is (no normalization, no dedup).
    - Diseases & chemicals: deduplicate by normalized text (case-insensitive).
    """
    entities: Dict[str, List[Dict[str, Any]]] = {
        "genes": [],
        "diseases": [],
        "chemicals": [],
    }

    # Track which normalized texts we've already added
    seen_diseases: set[str] = set()
    seen_chemicals: set[str] = set()
    seen_genes: set[str] = set()

    for passage in doc.get("passages", []):
        section = passage.get("infons", {}).get("section_type", "")
        for ann in passage.get("annotations", []):
            text = ann.get("text")
            if not text:
                continue

            infons = ann.get("infons", {})
            ent_type = infons.get("type")          # e.g. "Gene", "Disease", "Chemical"
            identifier = infons.get("identifier")  # e.g. "NCBIGene:672", "MESH:D009369"

            locs = ann.get("locations", [])
            start = locs[0]["offset"] if locs else None
            length = locs[0]["length"] if locs else None

            record = {
                "text": text,
                "identifier": identifier,
                "raw_type": ent_type,
                "section": section,
                "offset": start,
                "length": length,
            }

            # --- Genes: only type == "Gene", no Protein, no normalization/dedup ---
            if ent_type == "Gene":
                if text not in seen_genes:
                    seen_genes.add(text)
                    entities["genes"].append(record)

            # --- Diseases: dedup by normalized surface text ---
            elif ent_type == "Disease":
                norm = _normalize_mention(text)
                if norm not in seen_diseases:
                    seen_diseases.add(norm)
                    entities["diseases"].append(record)

            # --- Chemicals: dedup by normalized surface text ---
            elif ent_type in ("Chemical", "Drug"):
                norm = _normalize_mention(text)
                if norm not in seen_chemicals:
                    seen_chemicals.add(norm)
                    entities["chemicals"].append(record)

    return entities



def get_pubtator_entities_for_pmids(pmids: List[str], full: bool = True) -> Dict[str, Dict[str, list]]:
    """
    High-level helper:
    returns { pmid: {genes: [...], diseases: [...], chemicals: [...], variants: [...] } }
    """
    docs = fetch_pubtator_docs(pmids, full=full)
    results: Dict[str, Dict[str, list]] = {}

    for doc in docs:
        # PMID is usually stored in doc["infons"]["article-id_pmid"]
        infons = doc.get("infons", {})
        pmid = infons.get("article-id_pmid") or infons.get("pmid")

        if not pmid:
            # fallback: split _id like "29355051|PMC6142073"
            _id = doc.get("_id", "")
            pmid = _id.split("|")[0] if "|" in _id else _id

        results[str(pmid)] = parse_entities_from_doc(doc)

    return results


if __name__ == "__main__":
    entities_by_pmid = get_pubtator_entities_for_pmids(pmids, full=True)

    for pmid, ents in entities_by_pmid.items():
        print("PMID:", pmid)
        print("  Genes:")
        for g in ents["genes"]:
            print("    -", g["text"], g["identifier"])
        print("  Diseases:")
        for d in ents["diseases"]:
            print("    -", d["text"], d["identifier"])
        print("  Chemicals (candidate drugs):")
        for c in ents["chemicals"]:
            print("    -", c["text"], c["identifier"])
        print()


PMID: 19946579
  Genes:
    - renin 5972
    - parathyroid hormone 5741
    - erythropoietin 2056
    - insulin 3630
    - PTH 5741
  Diseases:
    - De Toni-Debre-Fanconi syndrome MESH:D005198
    - Kearns-Sayre syndrome MESH:D007625
    - pigmentary degeneration of the retina MESH:D007625
    - ophthalmoplegia MESH:D009886
    - mitochondrial myopathy MESH:D017240
    - glucosuria MESH:D006030
    - diabetes mellitus MESH:D003920
    - external ophthalmoplegia MESH:D009886
    - hyperaldosteronism MESH:D006929
    - sensorineural hearing impairment MESH:D006319
    - palpebral ptosis MESH:C564553
    - hyperaminoaciduria None
    - cardiac conduction defect MESH:D000075224
    - cerebellar ataxia MESH:D002524
    - hypoparathyroidism MESH:D007011
    - myopathy MESH:D009135
    - short stature MESH:D006130
    - hyperphosphaturia MESH:D007015
    - insufficiency of the renal tubule MESH:D051437
    - pigmentary retinopathy MESH:D012174
    - mitochondrial diseases MESH:D028361
    - 

In [32]:

from pathlib import Path

for pmcid in pmcids:
    title, abstract, para = get_pmc_fulltext(pmcid)
    input_dir = Path("../PhenoTagger_v1.2/input")
    input_dir.mkdir(parents=True, exist_ok=True)
    with open(input_dir /f"{pmcid}.PubTator", "w", encoding="utf-8") as f:
        f.write(f"{id}|t|{title}\n")
        f.write(f"{id}|a|{abstract}\n")
        for i in range(0, (len(para)-1)):
            f.write(f"{id}|p|{para[i]}\n")

In [7]:
from pathlib import Path
import json
from typing import Set, List, Dict, Any

def normalize_phenotype(text: str) -> str:
    """Normalize phenotype text for comparison (lowercase, strip whitespace)."""
    return " ".join(text.strip().lower().split())

def load_golden_phenotypes(pmcid: str, golden_dir: Path = Path("../golden")) -> Set[str]:
    """Load golden standard phenotypes from the txt file for a paper."""
    pheno_file = golden_dir / pmcid / "phenotype.txt"
    
    if not pheno_file.exists():
        print(f"Warning: phenotype file not found for {pmcid}")
        return set()
    
    phenotypes = set()
    with open(pheno_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                phenotypes.add(normalize_phenotype(line))
    
    return phenotypes

def extract_phenotypes_from_text(text: str, client: OpenAI, model: str = "medgemma") -> Set[str]:
    """Extract phenotypes from text using the LLM and return as normalized set."""
    from src.rarekg.prompts.entity_extraction import (
        extract_entities,
        PhenotypeExtractionResult
    )
    
    # Remove citations to clean text
    text_clean = strip_citation_like_bits(text)
    
    # Extract phenotypes using the LLM
    result = extract_entities(
        text_clean,
        client,
        model=model,
        extraction="phenotype",
        use_response_format=False,
    )
    
    # Convert to normalized set
    phenotypes = set()
    if isinstance(result, PhenotypeExtractionResult):
        for entity in result.entities:
            phenotypes.add(normalize_phenotype(entity.name))
    
    return phenotypes

def extract_phenotypes_from_paper(title: str, abstract: str, paragraphs: List[str], 
                                   client: OpenAI, model: str = "medgemma") -> Set[str]:
    """
    Extract phenotypes from a full paper by processing sections separately.
    Process title+abstract together, then each paragraph individually to stay within token limits.
    """
    all_phenotypes = set()
    
    # Process title + abstract together
    title_abstract = f"{title}\n{abstract}"
    try:
        phenos = extract_phenotypes_from_text(title_abstract, client, model=model)
        all_phenotypes.update(phenos)
        print(f"  Title+Abstract: extracted {len(phenos)} phenotypes")
    except Exception as e:
        print(f"  Warning: error processing title+abstract: {e}")
    
    # Process each paragraph individually
    for i, para in enumerate(paragraphs):
        try:
            phenos = extract_phenotypes_from_text(para, client, model=model)
            all_phenotypes.update(phenos)
            print(f"  Paragraph {i}: extracted {len(phenos)} phenotypes")
        except Exception as e:
            print(f"  Warning: error processing paragraph {i}: {e}")
    
    return all_phenotypes

def calculate_metrics(gold_set: Set[str], pred_set: Set[str]) -> Dict[str, float]:
    """Calculate precision, recall, and F1 score."""
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:


# Main evaluation loop
pmcids = ["PMC9437135", "PMC8901527", "PMC12021325", "PMC4289844", "PMC10032320", 
          "PMC10401045", "PMC9314610", "PMC9167473", "PMC7531107", "PMC2783042", 
          "PMC9278302", "PMC7702659", "PMC3014844", "PMC5717851", "PMC8488375"]

golden_dir = Path("../golden")
all_results = []

for pmcid in pmcids:
    print(f"\n{'='*60}")
    print(f"Processing: {pmcid}")
    print(f"{'='*60}")
    
    # Load golden phenotypes
    gold_phenotypes = load_golden_phenotypes(pmcid, golden_dir)
    
    if not gold_phenotypes:
        print(f"Skipping {pmcid} - no golden phenotypes found")
        continue
    
    # Get full paper
    try:
        title, abstract, paragraphs = get_pmc_fulltext(pmcid)
    except Exception as e:
        print(f"Error fetching paper {pmcid}: {e}")
        continue
    
    # Extract phenotypes section by section
    pred_phenotypes = extract_phenotypes_from_paper(title, abstract, paragraphs, client, model="medgemma")
    
    # Calculate metrics
    metrics = calculate_metrics(gold_phenotypes, pred_phenotypes)
    
    # Print results
    print(f"Golden phenotypes: {len(gold_phenotypes)}")
    print(f"Predicted phenotypes: {len(pred_phenotypes)}")
    print(f"True Positives: {metrics['tp']}")
    print(f"False Positives: {metrics['fp']}")
    print(f"False Negatives: {metrics['fn']}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1: {metrics['f1']:.4f}")
    
    # Show some examples
    if metrics['fp'] > 0:
        fp_examples = list(pred_phenotypes - gold_phenotypes)[:3]
        print(f"False Positive examples: {fp_examples}")
    
    if metrics['fn'] > 0:
        fn_examples = list(gold_phenotypes - pred_phenotypes)[:3]
        print(f"False Negative examples: {fn_examples}")
    
    all_results.append({
        "pmcid": pmcid,
        "metrics": metrics,
        "gold_phenotypes": list(gold_phenotypes),
        "pred_phenotypes": list(pred_phenotypes),
    })

# Calculate aggregate metrics
print(f"\n{'='*60}")
print("AGGREGATE RESULTS")
print(f"{'='*60}")

total_tp = sum(r["metrics"]["tp"] for r in all_results)
total_fp = sum(r["metrics"]["fp"] for r in all_results)
total_fn = sum(r["metrics"]["fn"] for r in all_results)

agg_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
agg_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
agg_f1 = (2 * agg_precision * agg_recall / (agg_precision + agg_recall)) if (agg_precision + agg_recall) > 0 else 0.0

print(f"Total Papers: {len(all_results)}")
print(f"Aggregate TP: {total_tp}")
print(f"Aggregate FP: {total_fp}")
print(f"Aggregate FN: {total_fn}")
print(f"Aggregate Precision: {agg_precision:.4f}")
print(f"Aggregate Recall: {agg_recall:.4f}")
print(f"Aggregate F1: {agg_f1:.4f}")

# Save results to file
results_file = Path("../phenotype_evaluation_results.json")
with open(results_file, "w", encoding="utf-8") as f:
    json.dump({
        "aggregate": {
            "precision": agg_precision,
            "recall": agg_recall,
            "f1": agg_f1,
            "total_tp": total_tp,
            "total_fp": total_fp,
            "total_fn": total_fn,
        },
        "per_paper": all_results,
    }, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to: {results_file}")



Processing: PMC9437135
  Title+Abstract: extracted 8 phenotypes
  Paragraph 0: extracted 15 phenotypes
  Paragraph 1: extracted 13 phenotypes
  Paragraph 2: extracted 25 phenotypes
  Paragraph 3: extracted 21 phenotypes
  Paragraph 4: extracted 18 phenotypes
  Paragraph 5: extracted 18 phenotypes
  Paragraph 6: extracted 3 phenotypes
  Paragraph 7: extracted 6 phenotypes
  Paragraph 8: extracted 6 phenotypes
  Paragraph 9: extracted 5 phenotypes
  Paragraph 10: extracted 9 phenotypes
  Paragraph 11: extracted 11 phenotypes
  Paragraph 12: extracted 14 phenotypes
  Paragraph 13: extracted 6 phenotypes
  Paragraph 14: extracted 11 phenotypes
  Paragraph 15: extracted 10 phenotypes
  Paragraph 16: extracted 6 phenotypes
  Paragraph 17: extracted 5 phenotypes
  Paragraph 18: extracted 7 phenotypes
  Paragraph 19: extracted 9 phenotypes
  Paragraph 20: extracted 4 phenotypes
  Paragraph 21: extracted 7 phenotypes
  Paragraph 22: extracted 2 phenotypes
  Paragraph 23: extracted 7 phenotypes

KeyboardInterrupt: 

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import importlib

def reload_entity_extraction():
    """Reload the entity_extraction module to pick up prompt changes."""
    import src.rarekg.prompts.entity_extraction as ee
    importlib.reload(ee)
    print("✓ entity_extraction module reloaded")

def visualize_paragraph_phenotypes(pmcid: str, client: OpenAI, model: str = "medgemma", golden_dir: Path = Path("../golden")):
    """
    Extract and visualize phenotypes from each paragraph of a paper.
    Shows side-by-side comparison with golden standard if available.
    """
    print(f"\n{'='*80}")
    print(f"DETAILED PARAGRAPH ANALYSIS: {pmcid}")
    print(f"{'='*80}\n")
    
    # Get full paper
    try:
        title, abstract, paragraphs = get_pmc_fulltext(pmcid)
    except Exception as e:
        print(f"Error fetching paper {pmcid}: {e}")
        return
    
    # Load golden phenotypes if available
    gold_phenotypes = load_golden_phenotypes(pmcid, golden_dir)
    
    # Results storage
    results = []
    
    # Process title
    print("="*80)
    print("TITLE")
    print("="*80)
    print(f"{title}\n")
    try:
        title_phenos = extract_phenotypes_from_text(title, client, model=model)
        print(f"Extracted {len(title_phenos)} phenotypes:")
        for pheno in sorted(title_phenos):
            in_gold = "✓ IN GOLD" if gold_phenotypes and pheno in gold_phenotypes else ""
            print(f"  • {pheno} {in_gold}")
        results.append({
            "section": "TITLE",
            "index": 0,
            "text_preview": title[:100] + "...",
            "num_extracted": len(title_phenos),
            "phenotypes": list(title_phenos),
        })
    except Exception as e:
        print(f"Error processing title: {e}")
    
    print()
    
    # Process abstract
    print("="*80)
    print("ABSTRACT")
    print("="*80)
    print(f"{abstract}\n")
    try:
        abstract_phenos = extract_phenotypes_from_text(abstract, client, model=model)
        print(f"Extracted {len(abstract_phenos)} phenotypes:")
        for pheno in sorted(abstract_phenos):
            in_gold = "✓ IN GOLD" if gold_phenotypes and pheno in gold_phenotypes else ""
            print(f"  • {pheno} {in_gold}")
        results.append({
            "section": "ABSTRACT",
            "index": 1,
            "text_preview": abstract[:100] + "...",
            "num_extracted": len(abstract_phenos),
            "phenotypes": list(abstract_phenos),
        })
    except Exception as e:
        print(f"Error processing abstract: {e}")
    
    print()
    
    # Process each paragraph
    for i, para in enumerate(paragraphs):
        print("="*80)
        print(f"PARAGRAPH {i}")
        print("="*80)
        print(f"{para[:200]}...\n")
        try:
            para_phenos = extract_phenotypes_from_text(para, client, model=model)
            print(f"Extracted {len(para_phenos)} phenotypes:")
            for pheno in sorted(para_phenos):
                in_gold = "✓ IN GOLD" if gold_phenotypes and pheno in gold_phenotypes else ""
                print(f"  • {pheno} {in_gold}")
            results.append({
                "section": f"PARAGRAPH {i}",
                "index": i + 2,
                "text_preview": para[:100] + "...",
                "num_extracted": len(para_phenos),
                "phenotypes": list(para_phenos),
            })
        except Exception as e:
            print(f"Error processing paragraph {i}: {e}")
        
        print()
    
    # Calculate aggregate metrics
    all_extracted_phenos = set()
    for r in results:
        all_extracted_phenos.update(r["phenotypes"])
    
    tp = len(all_extracted_phenos & gold_phenotypes) if gold_phenotypes else 0
    fp = len(all_extracted_phenos - gold_phenotypes) if gold_phenotypes else 0
    fn = len(gold_phenotypes - all_extracted_phenos) if gold_phenotypes else 0
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    
    # Print statistics
    print(f"{'='*80}")
    print("STATISTICS")
    print(f"{'='*80}")
    print(f"Total sections processed: {len(results)}")
    print(f"Unique phenotypes extracted: {len(all_extracted_phenos)}")
    
    if gold_phenotypes:
        print(f"\nGolden standard phenotypes: {len(gold_phenotypes)}")
        print(f"True Positives: {tp}")
        print(f"False Positives: {fp}")
        print(f"False Negatives: {fn}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
    
    # Plot metrics
    if gold_phenotypes:
        fig, ax = plt.subplots(figsize=(10, 6))
        metrics = ["Precision", "Recall", "F1-Score"]
        values = [precision, recall, f1]
        colors = ['#2ecc71', '#3498db', '#e74c3c']
        
        bars = ax.bar(metrics, values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        
        # Add value labels on bars
        for bar, value in zip(bars, values):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.4f}',
                   ha='center', va='bottom', fontsize=12, fontweight='bold')
        
        ax.set_ylabel("Score", fontsize=12, fontweight='bold')
        ax.set_title(f"Phenotype Extraction Metrics: {pmcid}", fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No golden standard available for comparison.")
    
    return results

# Example usage:
reload_entity_extraction()  # Call this after editing the prompt file
visualize_paragraph_phenotypes("PMC8901527", client)


✓ entity_extraction module reloaded

DETAILED PARAGRAPH ANALYSIS: PMC8901527

TITLE
Alexander disease: models, mechanisms, and medicine

Extracted 0 phenotypes:

ABSTRACT
Alexander disease is a primary disorder of astrocytes caused by gain of function mutations in the gene for GFAP, which lead to protein aggregation and a reactive astrocyte response, with devastating effects on the central nervous system. Over the past two decades since the discovery of GFAP as the culprit, several cellular and animal models have been generated, and much has been learned about underlying mechanisms contributing to the disease. Despite these efforts, many aspects of Alexander disease have remained enigmatic, particularly the initiating events in GFAP accumulation and astrocyte pathology, the relation between astrocyte dysfunction and myelin deficits, and the variability in age of onset and disease severity. More recent work in both old and new models has begun to address these complex questions and iden

KeyboardInterrupt: 

In [15]:
print(para[0])

Joubert syndrome (JS) is a rare congenital neurodevelopmental primary ciliopathy with a population‐based prevalence reaching 1.7 per 100,000 in the age range 0–19 years (Nuovo et al., 2020 ). First described by Dr Marie Joubert about 50 years ago (Joubert, Eisenring, Robb, & Andermann, 1969 ), JS is now diagnosed upon recognition of a pathognomonic malformation of the midbrain–hindbrain junction which results in the brain imaging finding “molar tooth sign” (MTS). This malformation, found in all patients, consists of cerebellar hypoplasia with vermian dysplasia, thick and horizontally oriented superior cerebellar peduncles, and an abnormally deep interpeduncular fossa (Maria et al., 1997 ). A spectrum of severity of the MTS has been reported (Poretti, Huisman, Scheer, & Boltshauser, 2011 ), and mild MTS presentations may be difficult to assess. Conversely, other cerebellar and brainstem malformations are sometimes wrongly interpreted as a mild MTS, leading to misdiagnosis (Aldinger et a